# Layers

> Potentially helpful layers for your models

In [ ]:
#| default_exp layers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, numpy as np, torch.nn.functional as F

from typing import Optional
from torch import nn
from torch import Tensor
from torch.nn.attention import SDPBackend, sdpa_kernel

## Linear Layers for Patches

In [ ]:
#| export
class PatchEncoder(nn.Module):
    def __init__(self, 
                 c_in, # the number of input channels
                 patch_len, # the length of the patches (either stft or interval length)
                 d_model, # the dimension of the initial linear layers for inputting patches into transformer
                 shared_embedding, # indicator of whether to project each channel individually or together
                 ):
        super().__init__()

        self.shared_embedding = shared_embedding
        self.n_vars = c_in
        self.patch_len = patch_len
        self.d_model = d_model

        # Input encoding: projection of feature vectors onto a d-dim vector space
        ## note that this could be an MLP too, if you want
        if not shared_embedding:
            self.W_P = nn.ModuleList()
            for _ in range(self.n_vars): self.W_P.append(nn.Linear(patch_len, d_model))
        else:
            self.W_P = nn.Linear(patch_len, d_model)

    def forward(self, x) -> Tensor:          
        """
        input: x: tensor [bs x num_patch x nvars x patch_len]
        returns: x: tensor [bs x num_patch x nvars x d_model]
        """
        # Input embedding
        if not self.shared_embedding:
            if x.is_nested:
                x_list = list(x.unbind())
                for i, x_i in enumerate(x_list):
                    x_out = []
                    for c in range(self.n_vars):
                        # Apply linear projection
                        x_out.append(self.W_P[c](x_i[:,c,:]))
                    x_list[i] = torch.stack(x_out, dim=1)
                x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
            else:
                x_out = []
                for i in range(self.n_vars):
                    x_out.append(self.W_P[i](x[:,:,i,:]))
                x = torch.stack(x_out, dim=2)
        else:
            x = self.W_P(x) # x: [bs x num_patch x nvars x d_model]
        return x

## Positional Encoding Layers

In [ ]:
#| export
class PositionalEncoding(nn.Module):
    def __init__(self, 
                 num_patch, # number of patches of time series or stft in input
                 d_model, # dimension of patch embeddings
                 #dropout=0.1 # dropout value
                 ):
        super().__init__()
        self.num_patch = num_patch
        self.d_model = d_model
        
        # Positional encoding - learned
        self.W_pos =  nn.Parameter(torch.empty((num_patch, d_model)))
        nn.init.uniform_(self.W_pos, -0.02, 0.02)
        self.scale_factor = nn.Parameter(torch.ones(1))
        #self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        """
        input: x: [bs * nvars x num_patch x d_model]
        returns: x: [bs * nvars x num_patch x d_model]
        """
        if x.is_nested:
            x_list = list(x.unbind())
            for i in range(len(x_list)):
                # Only use the portion of W_pos that matches this sequence length
                seq_len = x_list[i].size(0) # get number of patches
                x_list[i] = x_list[i] + self.scale_factor * self.W_pos[:seq_len]
            x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
        else:
            x = x + self.scale_factor * self.W_pos
        return x

In [ ]:
#| export
class tAPE(nn.Module):
    """
    time Absolute Position Encoding
    Adapted from tsai
    """

    def __init__(self, 
        d_model:int, # the embedding dimension
        seq_len:int, # the max. length of the incoming sequence or num patches
        ):
        super().__init__()
        self.scale_factor = nn.Parameter(torch.ones(1)) # learnable scale factor
        W_pos = torch.zeros(seq_len, d_model)  # positional encoding
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        W_pos[:, 0::2] = torch.sin((position * div_term)*(d_model/seq_len)) # this is the difference between normal PE and tAPE, scaling (d_model/seq_len)
        W_pos[:, 1::2] = torch.cos((position * div_term)*(d_model/seq_len))
        self.register_buffer('W_pos', W_pos)  # this stores the variable in the state_dict (used for non-trainable variables)
        #self.W_pos = self.scale_factor * self.W_pos.unsqueeze(0)
        

        #self.dropout = nn.Dropout(p=dropout)

    def forward(self, x): # [batch size, sequence length, embed dim]
        if x.is_nested:
            x_list = list(x.unbind())
            for i in range(len(x_list)):
                seq_len = x_list[i].size(0) # get number of patches
                x_list[i] = x_list[i] + self.W_pos[:seq_len] * self.scale_factor.to(x.device)
            x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
        else:
            x = x + self.W_pos.to(x.device) * self.scale_factor.to(x.device)
        return x

## Reversible Instance Normalization

In [ ]:
#| export
class RevIN(nn.Module):
    def __init__(self, 
                 num_features: int, # the number of channels or features in the input
                 eps=1e-5, # added to avoid division by zero errors
                 dim_to_reduce=-1, # the dimension to reduce, 
                 affine=True # learning affine parameters bias and weight per channel
                 ):
        """
        
        """
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        self.dim_to_reduce = dim_to_reduce

        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(num_features,1))
            self.affine_bias = nn.Parameter(torch.zeros(num_features,1))

    def forward(self, x, mode:bool):
        """
        x: [bs x n_vars x max_seq_len]
        """
        if mode:
            return self._normalize(x)
        else:
            return self._denormalize(x)

    def _normalize(self, x):
        if x.is_nested:
            was_nested = True
            seq_lens = [t.size(1) for t in x.unbind()]
            x = x.to_padded_tensor(0.)
        else:
            was_nested = False
        self.mean = torch.mean(x, dim=self.dim_to_reduce, keepdim=True).detach()
        self.stdev = torch.std(x, dim=self.dim_to_reduce, keepdim=True, unbiased=False).detach() + self.eps
        x = x.sub(self.mean)
        x = x.div(self.stdev)
        if self.affine:
            x = x.mul(self.affine_weight)
            x = x.add(self.affine_bias)
        if was_nested:
            out_tensors = [x[i, :, :seq_len] for i, seq_len in enumerate(seq_lens)]
            x = torch.nested.as_nested_tensor(out_tensors, layout=torch.jagged)
        return x

    def _denormalize(self, x):
        if x.is_nested:
            was_nested = True
            seq_lens = [t.size(1) for t in x.unbind()]
            x = x.to_padded_tensor(0.)
        else:
            was_nested = False
        if self.affine:
            x = x.sub(self.affine_bias)
            x = x.div(self.affine_weight)
        x = x.mul(self.stdev)
        x = x.add(self.mean)
        if was_nested:
            out_tensors = [x[i, :, :seq_len] for i, seq_len in enumerate(seq_lens)]
            x = torch.nested.as_nested_tensor(out_tensors, layout=torch.jagged)
        return x

## Attention

In [ ]:
#| export
class MultiheadFlashAttention(nn.Module):
    """Multihead attention layer with optional causal masking.
    Uses flash attention when available in PyTorch 2.0+.
    
    Args:
        n_heads (int): Number of attention heads
        d_model (int): Embedding dimension
        qkv_bias (bool, optional): Use bias in linear layers. Defaults to False.
        is_causal (bool, optional): Use causal masking. Defaults to False.
        attn_dropout (float, optional): Attention dropout probability. Defaults to 0.0.
        proj_dropout (float, optional): Dropout probability. Defaults to 0.0.

    Note that instead of a key padding mask, we can use a nested tensor to mask out the padding tokens.
    """
    _has_shown_fallback_warning = False # class level flag to show warning if flash attention is falling back
    def __init__(self, d_model: int, n_heads: int, qkv_bias: bool=True, 
                 is_causal: bool=False, attn_dropout: float=0.0, proj_dropout: float=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        
        # # key, query, value projections for all heads, but in a batch, Combined Q,K,V projections
        self.W_Q = nn.Linear(d_model, d_model, bias=qkv_bias)
        self.W_K = nn.Linear(d_model, d_model, bias=qkv_bias)
        self.W_V = nn.Linear(d_model, d_model, bias=qkv_bias)
        
        # Output projection
        self.c_proj = nn.Linear(d_model, d_model, bias=qkv_bias)
        head_dim = d_model // n_heads
        self.scale = head_dim ** -0.5
        
        # Regularization
        self.attn_dropout = attn_dropout
        self.resid_dropout = nn.Dropout(proj_dropout)
        
        # Architecture
        self.num_heads = n_heads
        self.embed_dimension = d_model
        self.is_causal = is_causal

    def forward(self, x: Tensor, channel_mask: Optional[Tensor] = None) -> Tensor:
        """
        x: [bs x seq_len x d_model]
        """
        head_dim = self.embed_dimension // self.num_heads
        is_nested = x.is_nested

        if is_nested and channel_mask is not None:
            # if a channel mask is provided, we need to pad the input tensor to the maximum sequence length
            # and combine it with a channel mask
            seq_lens = [t.size(0) for t in x.unbind()]
            max_len = max(seq_lens)
            batch_channel_size = len(seq_lens)
            
            # Create padding mask where True indicates padding positions
            attn_mask = torch.zeros(batch_channel_size, max_len, dtype=torch.bool, device=x.device) # [bs * n_channels x n_patches]
            channel_mask = channel_mask.reshape(-1) # [bs * n_channels]
            for i, length in enumerate(seq_lens):
                attn_mask[i, length:] = True
                if channel_mask[i]:
                    attn_mask[i] = True

            # Convert to padded tensor
            x = x.to_padded_tensor(0.)
            attn_mask = attn_mask[:, None, None, :] # [bs * n_channels x 1 x 1 x n_patches]
        elif not is_nested and channel_mask is not None:
            # Reshape channel mask to match attention shape
            # [bs x n_channels] -> [bs * n_channels x n_patches])
            channel_mask = channel_mask.reshape(-1)[:, None].expand(-1, x.size(1))
            attn_mask = channel_mask[:, None, None, :] # [bs * n_channels x 1 x 1 x n_patches]
        else:
            attn_mask = None
        # Split and reshape
        query = self.W_Q(x).unflatten(-1, [self.num_heads, head_dim]).transpose(1, 2)
        key = self.W_K(x).unflatten(-1, [self.num_heads, head_dim]).transpose(1, 2)
        value = self.W_V(x).unflatten(-1, [self.num_heads, head_dim]).transpose(1, 2)

        # Set dropout and causality based on training mode
        attn_dropout = self.attn_dropout if self.training else 0.0
        is_causal = self.is_causal if self.training else False
        # Handle channel masking if provided
        # Attention (with flash attention when available)
        try:
            with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
                y = F.scaled_dot_product_attention(
                    query, key, value, 
                    attn_mask=attn_mask,  # PyTorch handles mask preprocessing internally, can pass attn_mask or key_padding_mask
                    dropout_p=attn_dropout, 
                    is_causal=is_causal,
                    scale=self.scale
                )
        except RuntimeError as e:
            if not MultiheadFlashAttention._has_shown_fallback_warning:
                import warnings
                warnings.warn(f"Flash Attention not available. Falling back to memory efficient or math attention. Error: {e}")
                MultiheadFlashAttention._has_shown_fallback_warning = True
            with sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH]):
                y = F.scaled_dot_product_attention(
                    query, key, value, 
                    attn_mask=attn_mask,
                    dropout_p=attn_dropout, 
                    is_causal=is_causal,
                    scale=self.scale
                )
        
        # Reshape and project output
        y = y.transpose(1, 2).flatten(-2)
        y = self.resid_dropout(self.c_proj(y))

        if is_nested and channel_mask is not None:
            out_tensors = [y[i, :seq_len] for i, seq_len in enumerate(seq_lens)]
            y = torch.nested.as_nested_tensor(out_tensors, layout=torch.jagged)
        return y

In [ ]:
#| notest
mha = MultiheadFlashAttention(d_model=512, n_heads=8, attn_dropout=0., proj_dropout=0.)
x = torch.randn(2*7,100,512) # [bs * n_vars x n_patches x d_model]
key_padding_mask = torch.zeros(2*7, 100, dtype=torch.bool)
key_padding_mask[:, -2:] = True  # mask last 2 positions
output = mha(x, key_padding_mask=key_padding_mask)
output.shape

torch.Size([14, 100, 512])

## Miscellaneous

In [ ]:
#| export
class LearnableMaskedChannelTokens(nn.Module):
    def __init__(self, missing_channel_indices, d_model):
        super().__init__()
        # Create learnable tokens for each channel and each potential patch position
        self.missing_channel_indices = missing_channel_indices
        # create learnable tokens for each missing channel, but not per patch position
        self.mask_token = nn.Parameter(torch.zeros(len(missing_channel_indices), d_model, 1))
        torch.nn.init.normal_(self.mask_token, std=.02)
        
    def forward(self, x):
        """
        x: Input tensor [bs x nvars x d_model x num_patch] 
        """
        if x.is_nested:
            x_list = list(x.unbind())
            for i, x_i in enumerate(x_list):
                t = x_i.clone()
                t[self.missing_channel_indices] = t[self.missing_channel_indices] + self.mask_token
                x_list[i] = t
            x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
        else:
            x[:, self.missing_channel_indices] = x[:, self.missing_channel_indices] + self.mask_token
        return x


In [ ]:
#| export
class Identity(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x, **kwargs):
        return x

In [ ]:
#| export
class Transpose(nn.Module):
    def __init__(self, *dims, contiguous=False): 
        super().__init__()
        self.dims, self.contiguous = dims, contiguous
    def forward(self, x):        
        if self.contiguous: return x.transpose(*self.dims).contiguous()
        else: return x.transpose(*self.dims)


def get_activation_fn(activation):
    if callable(activation): return activation()
    elif activation.lower() == "relu": return nn.ReLU()
    elif activation.lower() == "gelu": return nn.GELU()
    raise ValueError(f'{activation} is not available. You can use "relu", "gelu", or a callable')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()